<a href="https://colab.research.google.com/github/bsheese/387/blob/main/20_how_networks_learn/20_0_PyTorch_Basics/20_0_2_Autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How Networks Learn: Part 2
## The Nudge Test and Autograd

---

## What This Notebook Is About

Training a network means changing its weights so that the loss gets smaller. In 19_0_1 and 19_1_1 that happened inside two lines of the training loop, `loss.backward()` and `optimizer.step()`. This notebook explains the first of them.

To improve a weight, you need to know two things: which way to move it (up or down), and how strongly the loss responds to it. Both come from one number, the weight's **slope**: how much the loss changes when the weight changes by a tiny amount. There's a direct way to measure a slope, which we'll call the **nudge test**:

1. Compute the loss.
2. Nudge one weight by a tiny amount and compute the loss again.
3. Divide the change in the loss by the size of the nudge.

The nudge test needs no calculus, and it always works. Its problem is cost: it takes extra runs of the model for every single weight. PyTorch's **autograd** gets the same numbers for every weight at once, in roughly the time of a few runs. This notebook measures slopes with the nudge test and then checks that autograd agrees.

**What you will learn:**
1. How to measure a slope with the nudge test, and what its sign and size mean
2. That autograd (`.backward()` and `.grad`) gives the same slopes
3. How slopes combine through several steps of a calculation ("rates multiply")
4. Why networks can't be trained by nudging, measured on layers of different sizes
5. Two habits the training loop depends on: zeroing gradients, and `torch.no_grad()`

If you've taken calculus, a slope here is a derivative, and boxes marked "If you've taken calculus" connect the two. You don't need them for anything in the course.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

torch.manual_seed(0)
print('PyTorch version:', torch.__version__)

---

## Section 1: The Nudge Test

Start with something simpler than a network: $y = x^2$. At $x = 3$, $y = 9$. How fast does $y$ change when $x$ changes? Nudge $x$ up by a tiny amount and see:

In [ ]:
def f(x):
    return x ** 2

x = 3.0
h = 0.001                          # the size of the nudge
print('f(3)     =', f(x))
print('f(3.001) =', f(x + h))
print('change in y divided by change in x:', (f(x + h) - f(x)) / h)

Nudging $x$ up by 0.001 makes $y$ go up by about 0.006, six times as much. So at $x = 3$, the slope of $y = x^2$ is about 6.

The answer, 6.001, is slightly off because the nudge only went one way. Nudging both up and down and dividing by the total distance (2 × 0.001) gives a more accurate slope. This is the version we'll use from now on:

In [ ]:
def nudge_slope(f, x, h=0.001):
    # the nudge test: change in f across a tiny step either side of x, divided by the step
    return (f(x + h) - f(x - h)) / (2 * h)

print('slope at x = 3:', round(nudge_slope(f, 3.0), 6))

The slope depends on where you are. Here it is at several values of $x$, drawn as short lines that touch the curve:

In [ ]:
xs = np.linspace(-4, 4, 200)
plt.figure(figsize=(7, 4))
plt.plot(xs, f(xs), label='y = x²')
for x0 in [-3, -1, 0, 2, 3]:
    s = nudge_slope(f, x0)
    near = np.linspace(x0 - 0.8, x0 + 0.8, 10)
    plt.plot(near, f(x0) + s * (near - x0), 'r-', lw=2)
    plt.annotate(f'slope {s:.0f}', (x0, f(x0)), xytext=(5, 12), textcoords='offset points', fontsize=9)
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

How to read a slope:

- **Its sign** says which way is uphill. At $x = 3$ the slope is positive, so increasing $x$ increases $y$. At $x = -3$ it's negative, so increasing $x$ decreases $y$.
- **Its size** says how strongly $y$ responds. The slope is 4 at $x = 2$ and 6 at $x = 3$: the curve gets steeper the farther you go from 0.
- **A slope of 0** means $y$ doesn't respond to small changes at all. For $y = x^2$ that happens at the very bottom, $x = 0$.

If $y$ were a loss and $x$ a weight, the slope would tell us what to do: move the weight in the direction opposite to the sign of the slope, since that is downhill, by an amount that depends on the slope's size. That's gradient descent, the subject of 20_0_3.

---

## Section 2: Autograd Gets the Same Number

Now the same calculation with PyTorch. Make `x` a tensor that tracks how it is used (`requires_grad=True`, from 20_0_1), compute `y`, and call `y.backward()`. PyTorch then stores the slope of `y` with respect to `x` in `x.grad`. PyTorch calls this slope the **gradient**.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()
print('x.grad:', x.grad.item())
print('nudge test:', round(nudge_slope(f, 3.0), 6))

They agree. `.backward()` didn't nudge anything. It used the record PyTorch kept while computing `y` (the `grad_fn` from 20_0_1) to work out the slope directly. That's **autograd**, short for automatic gradients.

<details>
<summary><b>If you've taken calculus</b></summary>

The slope measured by the nudge test is the derivative. For $y = x^2$, $\frac{dy}{dx} = 2x$, which is 6 at $x = 3$. The nudge test with $h = 0.001$ is the difference quotient $\frac{f(x+h) - f(x-h)}{2h}$, which approaches the derivative as $h$ shrinks. Autograd doesn't use difference quotients. It applies the derivative rule for each operation it recorded (here, the power rule), so its answer is exact up to rounding.
</details>

### Slopes of a loss with respect to weights

Here is the situation training is actually in: a single neuron with two weights and a bias, one example with a target value, and a squared-error loss. (Squared error is a common loss when the target is a number: square the difference between the prediction and the target.) We want the slope of the loss with respect to each of the three parameters.

First with the nudge test, one parameter at a time. Each nudge needs two extra runs of the calculation:

In [ ]:
x1, x2, target = 3.0, 2.0, 1.0

def loss_of(w1, w2, bias):
    prediction = w1 * x1 + w2 * x2 + bias
    return (prediction - target) ** 2

w1, w2, bias = 2.0, 0.5, 0.3
print('prediction:', w1 * x1 + w2 * x2 + bias, '  loss:', round(loss_of(w1, w2, bias), 4))
print('slope for w1:  ', round(nudge_slope(lambda v: loss_of(v, w2, bias), w1), 4))
print('slope for w2:  ', round(nudge_slope(lambda v: loss_of(w1, v, bias), w2), 4))
print('slope for bias:', round(nudge_slope(lambda v: loss_of(w1, w2, v), bias), 4))

(`lambda v: loss_of(v, w2, bias)` is a one-line function of a single number `v`: the loss when `w1` is replaced by `v` and the other two stay put. It lets `nudge_slope` nudge one parameter at a time.)

Now with autograd. One call to `.backward()` fills in `.grad` for all three:

In [ ]:
w1 = torch.tensor(2.0, requires_grad=True)
w2 = torch.tensor(0.5, requires_grad=True)
bias = torch.tensor(0.3, requires_grad=True)

loss = (w1 * x1 + w2 * x2 + bias - target) ** 2
loss.backward()
print('w1.grad:  ', round(w1.grad.item(), 4))
print('w2.grad:  ', round(w2.grad.item(), 4))
print('bias.grad:', round(bias.grad.item(), 4))

The same three numbers. All three slopes are positive, so increasing any of the parameters would increase the loss. That makes sense: the prediction (7.3) is far above the target (1.0), so every parameter should go down. The slope for `w1` is the largest because `w1` multiplies the largest input (3.0), so changes to it move the prediction the most.

---

## Section 3: Rates Multiply

A network is a long chain of calculations: the weights produce a weighted sum, the sum goes through an activation, that feeds the next layer, and so on until the loss. How does a slope get from the loss all the way back to a weight at the start of the chain?

Take a two-step calculation with $a = 2$, $b = 1$ and $x = 3$:

- step 1: $u = a \cdot x + b$, which is 7
- step 2: $z = u^2$, which is 49

We want the slope of $z$ with respect to $a$. Measure each step's slope separately with the nudge test:

In [ ]:
a, b, x = 2.0, 1.0, 3.0
u = a * x + b

rate_u_per_a = nudge_slope(lambda v: v * x + b, a)    # how fast u changes when a changes
rate_z_per_u = nudge_slope(lambda v: v ** 2, u)       # how fast z changes when u changes
print('u changes', round(rate_u_per_a, 4), 'times as much as a')
print('z changes', round(rate_z_per_u, 4), 'times as much as u')
print('product:  ', round(rate_u_per_a * rate_z_per_u, 4))

A nudge to $a$ moves $u$ 3 times as much, and a nudge to $u$ moves $z$ 14 times as much. So a nudge to $a$ should move $z$ 3 × 14 = 42 times as much. Checking that by nudging $a$ directly, and with autograd:

In [ ]:
print('nudge a, measure z:', round(nudge_slope(lambda v: (v * x + b) ** 2, a), 4))

a_t = torch.tensor(2.0, requires_grad=True)
z = (a_t * x + b) ** 2
z.backward()
print('autograd:          ', a_t.grad.item())

All three give 42. **Rates multiply**: the slope through a chain of steps is the product of the slopes of the individual steps. This is how autograd handles a whole network. It starts at the loss and works backward through the recorded steps, multiplying the slope of each step as it goes. Each step's slope is worked out once and shared by every weight that comes before it, so the whole backward pass takes roughly as long as running the network forward two or three times, however many weights there are. Working backward this way is called **backpropagation**.

<details>
<summary><b>If you've taken calculus</b></summary>

"Rates multiply" is the chain rule: $\frac{dz}{da} = \frac{dz}{du} \cdot \frac{du}{da} = 2u \cdot x = 14 \cdot 3 = 42$. Backpropagation is the chain rule applied from the output backward, reusing each $\frac{d\,\text{loss}}{d\,(\text{intermediate})}$ for everything upstream of it. Calculating in this order is also called reverse-mode automatic differentiation.
</details>

---

## Section 4: Why Not Just Nudge Every Weight?

The nudge test is simple and always right, up to rounding. So why does PyTorch need autograd? Because of cost. The nudge test needs two extra runs of the model **per weight**, while `.backward()` gets every weight's slope in one pass.

The next cell measures both on layers of different sizes. Each layer is a set of neurons, as in 19_1_3, with a squared-error loss on a batch of 64 made-up examples. For the nudge test it nudges every weight up and down in turn; for autograd it calls `.backward()` once. It uses 64-bit numbers (`.double()`) so that the tiny nudges aren't lost to rounding, which lets us check that both methods agree.

In [ ]:
def time_both(n_inputs, n_outputs):
    layer = nn.Linear(n_inputs, n_outputs).double()
    X = torch.randn(64, n_inputs, dtype=torch.float64)
    Y = torch.randn(64, n_outputs, dtype=torch.float64)
    def loss_fn():
        return ((layer(X) - Y) ** 2).mean()

    # the nudge test, one weight at a time
    start = time.time()
    nudged = []
    with torch.no_grad():
        for p in layer.parameters():
            values = p.view(-1)              # every weight in this tensor, in one row
            for i in range(values.numel()):
                original = values[i].item()
                values[i] = original + 1e-4
                up = loss_fn().item()
                values[i] = original - 1e-4
                down = loss_fn().item()
                values[i] = original
                nudged.append((up - down) / 2e-4)
    nudge_seconds = time.time() - start

    # autograd, all at once
    start = time.time()
    loss_fn().backward()
    autograd_seconds = time.time() - start
    from_autograd = torch.cat([p.grad.view(-1) for p in layer.parameters()]).numpy()

    n = sum(p.numel() for p in layer.parameters())
    gap = np.abs(np.array(nudged) - from_autograd).max()
    print(f'{n:7,} weights   nudge test {nudge_seconds:7.3f} s   autograd {autograd_seconds:7.4f} s   '
          f'largest disagreement {gap:.1e}')

for n_inputs, n_outputs in [(50, 20), (200, 50), (500, 100)]:
    time_both(n_inputs, n_outputs)

The two methods agree to many decimal places. But the nudge test's time grows with the number of weights, and autograd's barely changes. (Exact times depend on the computer, and very short times are mostly overhead.)

Now scale that up. ResNet-18, the model from 19_0, has about 11.7 million weights. The nudge test would need about 23 million runs of the network for a single training step. Even at a hundredth of a second per run, that's about 234,000 seconds, nearly three days, to take one step downhill. Autograd gets the same 11.7 million slopes in roughly the time of two or three runs. Without it, training networks of any real size wouldn't be practical.

The nudge test is still useful. It's the standard way to check that a gradient is right: if a hand-written calculation and autograd disagree with the nudge test, something is wrong.

---

## Section 5: Two Habits the Training Loop Depends On

### Gradients add up until you clear them

Calling `.backward()` doesn't replace what's in `.grad`; it adds to it. Watch what happens when the same calculation runs three times:

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
for step in range(1, 4):
    y = x ** 2
    y.backward()
    print(f'step {step}: x.grad = {x.grad.item()}')

The slope of $x^2$ at $x = 3$ is 6 every time, as the nudge test showed, but `.grad` reads 6, 12, 18 because each call adds another 6. Adding up is useful in a few special cases, but in an ordinary training loop it's a bug: every step would use a mixture of old and new slopes. So every step starts by setting the gradients back to zero.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
for step in range(1, 4):
    if x.grad is not None:
        x.grad.zero_()          # clear last step's slope (the underscore means in place)
    y = x ** 2
    y.backward()
    print(f'step {step}: x.grad = {x.grad.item()}')

In a training loop, `optimizer.zero_grad()` does this for every weight in the model at once.

### Turning tracking off when you don't need slopes

Keeping the record that `.backward()` uses costs memory and time. When you only want the model's outputs, such as checking validation accuracy or making predictions, wrap the calculation in `with torch.no_grad():`. Nothing inside it is tracked.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2
print('tracked:    ', y.grad_fn)

with torch.no_grad():
    y = x ** 2
print('not tracked:', y.grad_fn)

You saw `with torch.no_grad():` in 19_0_1's `accuracy` and `predict` functions, which only needed outputs.

---

## Section 6: Back to the Training Loop

Here are the lines of 19_0_1's training loop that you can now read:

```python
scores = model(x)          # the forward pass: PyTorch records every calculation
loss = loss_fn(scores, y)  # one number at the end of the chain
optimizer.zero_grad()      # clear the slopes from the last step (Section 5)
loss.backward()            # autograd: the slope of the loss for every trainable weight (Sections 2-4)
optimizer.step()           # move every weight a little downhill
```

The only line left is `optimizer.step()`, which uses the slopes to change the weights. 20_0_3 writes that step by hand.

---

## Summary

- A weight's **slope** (its **gradient**) is how much the loss changes for a tiny change in the weight. Its sign says which way is uphill, and its size says how strongly the loss responds.
- The **nudge test** measures a slope directly: change one value a tiny amount up and down, and divide the change in the result by the total change. It needs no calculus.
- **Autograd** gets the same slopes. After `loss.backward()`, every tracked tensor's `.grad` holds its slope.
- **Rates multiply**: the slope through a chain of steps is the product of each step's slope. Working backward from the loss this way is **backpropagation**, and it costs about as much as a few passes through the network.
- The nudge test needs two extra passes per weight, which is millions of passes for ResNet-18, so training relies on autograd. The nudge test remains the standard check that a gradient is right.
- `.grad` adds up across `.backward()` calls, so every training step starts with `optimizer.zero_grad()`. `torch.no_grad()` turns tracking off when only outputs are needed.

**Next:** 20_0_3 uses slopes to find good weights: gradient descent, written from scratch on a small regression problem.